<a href="https://colab.research.google.com/github/COL-O/POTOK/blob/main/notebooks/potok-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# POTOK Colab: полная верификация `main` (вне Actions)

**Схема та же, что и раньше, плюс доказанный обход занятых портов.** Одна тетрадь — полный круг проверок `main`:

1. свежий клон `main` →
2. Go 1.22.12 официальным архивом →
3. `go test -race -count=1 ./...` →
4. 5× бывший флейк фронта (`TestRouterRoutineVetoAndQuarantine`) →
5. `go build` + `examples/simple` + `examples/drone` →
6. организм 4 такта — **через `unshare -n`** (в Colab порты `:8080`/`:9000` заняты инфраструктурой: `node` pid 6, `jupyter-server` pid 120 — их не трогаем) →
7. автоматическая сверка «распечатка ↔ метрики» (КОН 2).

Эталон на `main ≥ 832e3d0` (после PR #24): все тесты `ok`; флейк 5/5 PASS; организм rc 0; такт 3 — **Gen#2 Guard=[-5.1..31.2], BlackSwans=1/1, Alpha=0.1**; тепловой отчёт `Новых законов=1 | Реабилитаций Veto=1 | WAL=1` — распечатка и метрики сходятся.

⚠️ Секретов нет; тетрадь никуда ничего не отправляет; записывает только в `/content`. Итоги — в журналы дома (LOGBOOK/RUN-LOG) руками человека.

In [ ]:
# 1. Свежий клон main
import os, subprocess
REPO = '/content/potok-run'
subprocess.run(['rm', '-rf', REPO], check=False)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main',
                'https://github.com/COL-O/POTOK.git', REPO], check=True)
print(subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)


In [ ]:
# 2. Go 1.22.12 официальным архивом
import os, subprocess
subprocess.run('curl -sL https://dl.google.com/go/go1.22.12.linux-amd64.tar.gz | tar -C /usr/local -xz',
               shell=True, check=True)
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
print(subprocess.run(['go', 'version'], capture_output=True, text=True).stdout)


In [ ]:
# 3. Иммунитет: полный прогон с гонками (как шаг CI)
import os, subprocess
REPO = '/content/potok-run'
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
r = subprocess.run(['go', 'test', '-race', '-count=1', './...'],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.stderr:
    print(r.stderr[-1500:])
print('ВЕРДИКТ ТЕСТОВ:', 'PASS' if r.returncode == 0 else f'FAIL (код {r.returncode})')


In [ ]:
# 4. Бывший флейк фронта: 5 подряд
import os, subprocess
REPO = '/content/potok-run'
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
for i in range(1, 6):
    r = subprocess.run(['go', 'test', '-race', '-count=1',
                        '-run', 'TestRouterRoutineVetoAndQuarantine', './modules/front/'],
                       cwd=REPO, capture_output=True, text=True)
    print(f'флейк-прогон {i}:', 'PASS' if r.returncode == 0 else 'FAIL')


In [ ]:
# 5. Сборка и два примера
import os, subprocess
REPO = '/content/potok-run'
os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']
subprocess.run(['go', 'build', '-o', 'bin/organism', './cmd/organism/...'], cwd=REPO, check=True)
print('--- examples/simple')
subprocess.run(['go', 'run', './examples/simple/main.go'], cwd=REPO, check=True)
print('--- examples/drone')
subprocess.run(['go', 'run', './examples/drone/main.go'], cwd=REPO, check=True)


In [ ]:
# 6. Организм: 4 такта в изолированном netns (обход занятого Colab'ом :8080/:9000)
import os, subprocess
REPO = '/content/potok-run'
r = subprocess.run(['unshare', '-n', REPO + '/bin/organism'], cwd=REPO,
                   capture_output=True, text=True, timeout=180)
print(r.stdout)
if r.stderr:
    print('--- stderr ---')
    print(r.stderr)
print('КОД ВОЗВРАТА ОРГАНИЗМА:', r.returncode)


In [ ]:
# 7. Сверка распечатки и метрик (КОН 2): закон выведен — значит, он и в учёте
import os, subprocess
REPO = '/content/potok-run'
r = subprocess.run(['unshare', '-n', REPO + '/bin/organism'], cwd=REPO,
                   capture_output=True, text=True, timeout=180)
out = r.stdout
print(out)
import re
m_laws = re.search(r'Новых законов=(\d+)', out)
m_wal  = re.search(r'WAL=(\d+)', out)
m_claim = 'Выведен новый закон' in out
laws = int(m_laws.group(1)) if m_laws else -1
wal  = int(m_wal.group(1)) if m_wal else -1
print(f'распечатка: «Выведен новый закон» = {m_claim}; метрики: Новых законов={laws}, WAL={wal}')
same = (m_claim == (laws > 0)) and (laws == wal)
print('СВЁРКА:', 'СОГЛАСНО — распечатка и метрики сходятся' if same else 'РАСХОЖДЕНИЕ — факт для задачи №6')
print('ИТОГ PR-24/MAIN:', 'ЗЕЛЁНО' if (same and r.returncode == 0) else 'КРАСНО')


## Куда сообщать

Вывод ячеек → репликой в №12 (через мост: `inbox/<имя>.md` + push) или строки `ВЕРДИКТ ТЕСТОВ`, `СВЁРКА`, `ИТОГ PR-24/MAIN` — задачи №6/#12. Если Colab занят своими портами — шаг 6 уже это учитывает; ничего убивать не надо.

*Протокол верификации `main` после состыковки PR #24; зафиксирован основателем 2026-11-20. Эталон неизменности: CI main зелёный (run 44788285312), CodeQL pass.*